## Question1

MapReduce limitations:
1. Disk I/O after every step — MapReduce writes intermediate results to HDFS after each Map and Reduce phase. Multi-step jobs read and write disk repeatedly, making them extremely slow.
2. No in-memory caching — There is no way to reuse data across iterations. Every job starts fresh from disk.
3. Only two phases (Map & Reduce) — Complex logic requires chaining multiple MapReduce jobs, each with its own disk write overhead.
4. High latency — Not suitable for interactive queries or real-time processing. Each job takes minutes to initialize.
5. No native streaming — MapReduce is batch-only. Spark supports batch, streaming, SQL, and ML in one unified engine.
6. Verbose code — Simple operations require writing verbose Java/Python boilerplate.


Why Spark wins: Spark keeps data in-memory across stages, supports lazy evaluation, and has a rich API (DataFrames, SQL, MLlib, Streaming) — all in one framework.

## Question 2

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("IterativeML").getOrCreate()

# Load dataset once
df = spark.read.parquet("s3://bucket/training_data/")

# Cache in memory — subsequent iterations reuse this
df.cache()

# Simulate 10 training iterations
for epoch in range(10):
    result = df.groupBy("label").agg({"feature": "mean"}).collect()
    print(f"Epoch {epoch+1} done")

# Free memory when done
df.unpersist()

## Question 3

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Dedup").getOrCreate()

# Sample DataFrame
df = spark.createDataFrame([
    (1, "2024-01-01", 500.0),
    (1, "2024-01-01", 500.0),   # duplicate
    (2, "2024-01-02", 300.0),
    (3, "2024-01-01", 700.0),
], ["user_id", "transaction_date", "amount"])

# Remove duplicates based on specific columns
df_clean = df.dropDuplicates(["user_id", "transaction_date"])

df_clean.show()

## Question 4

In [ ]:
from pyspark.sql import functions as F

# Filter for 'West' region, then group by product_category
result = (
    df_sales
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))
    .orderBy("avg_sale_amount", ascending=False)
)

result.show()
# Output example:
# +----------------+---------------+
# |product_category|avg_sale_amount|
# +----------------+---------------+
# |     Electronics|         1250.5|
# |        Clothing|          340.2|
# |       Furniture|          890.0|
# +----------------+---------------+

## Question 5

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("NullHandling").getOrCreate()

# Sample data with nulls
df = spark.createDataFrame([
    (1, "active"),
    (2, None),
    (3, "inactive"),
    (4, None),
], ["user_id", "status"])

df.show()
# +-------+--------+
# |user_id|  status|
# +-------+--------+
# |      1|  active|
# |      2|    null|
# |      3|inactive|
# |      4|    null|
# +-------+--------+

# .na.drop() — removes rows with any null
df_dropped = df.na.drop()
df_dropped.show()
# Only rows 1 and 3 remain

# .na.fill() — fills nulls with a value in 'status' column
df_filled = df.na.fill("Unknown", subset=["status"])
df_filled.show()
# +-------+--------+
# |user_id|  status|
# +-------+--------+
# |      1|  active|
# |      2| Unknown|
# |      3|inactive|
# |      4| Unknown|
# +-------+--------+

## Question 6

In [ ]:
from pyspark.sql import functions as F

result = (
    df
    .groupBy("city")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 100)
    .orderBy("record_count", ascending=False)
)

result.show()
# +------------+------------+
# |        city|record_count|
# +------------+------------+
# |      Mumbai|         450|
# |       Delhi|         320|
# |   Bangalore|         210|
# +------------+------------+

## Question 7

In [ ]:
# WRONG mental model — this does NOT modify df in place
df.drop("unnecessary_column")          # df is unchanged
df.withColumnRenamed("old", "new")     # df is unchanged

# CORRECT — always assign the result to a variable
df = df.drop("unnecessary_column")
df = df.withColumnRenamed("old_name", "new_name")

# Better pattern — chain transformations
df_clean = (
    df
    .drop("unnecessary_column", "another_col")
    .withColumnRenamed("old_name", "new_name")
    .filter(F.col("age").isNotNull())
    .dropDuplicates(["user_id"])
)

Benefits of immutability:


Enables fault tolerance — Spark can rebuild any DataFrame from its lineage (DAG)
Makes debugging easier — you can inspect intermediate DataFrames
Enables lazy evaluation — Spark optimizes the whole chain before executing

## Question 8

In [ ]:
from pyspark.sql import functions as F

df_filtered = df.filter(
    (F.col("age").between(18, 30)) &
    (F.col("subscription") == "Premium")
)

df_filtered.show()

## Question 9

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("NullAgg").getOrCreate()

df = spark.createDataFrame([
    (1, 100.0),
    (2, 200.0),
    (3, None),     # null price
    (4, 300.0),
], ["id", "price"])

# avg() silently ignores the null — calculates avg of 3 values, not 4
df.agg(F.avg("price")).show()
# Result: 200.0 (average of 100, 200, 300 — null skipped)

# What you might WANT: treat null as 0
df_fixed = df.na.fill(0, subset=["price"])
df_fixed.agg(F.avg("price")).show()
# Result: 150.0 (average of 100, 200, 0, 300)

## Question 10

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("NullAgg").getOrCreate()

df = spark.createDataFrame([
    (1, 100.0),
    (2, 200.0),
    (3, None),     # null price
    (4, 300.0),
], ["id", "price"])

# avg() silently ignores the null — calculates avg of 3 values, not 4
df.agg(F.avg("price")).show()
# Result: 200.0 (average of 100, 200, 300 — null skipped)

# What you might WANT: treat null as 0
df_fixed = df.na.fill(0, subset=["price"])
df_fixed.agg(F.avg("price")).show()
# Result: 150.0 (average of 100, 200, 0, 300)

## Question 11

Shuffle = moving data across the network so matching keys land on the same machine.

Before shuffle (data spread across 3 executors):
Executor 1: [Mumbai, Delhi, Chennai, Mumbai]
Executor 2: [Delhi, Mumbai, Bangalore, Chennai]
Executor 3: [Bangalore, Delhi, Mumbai, Bangalore]

After shuffle (grouped by city):
Executor 1: [Mumbai, Mumbai, Mumbai, Mumbai]  ← all Mumbai rows
Executor 2: [Delhi, Delhi, Delhi]             ← all Delhi rows
Executor 3: [Chennai, Chennai, Bangalore, Bangalore, Bangalore]

Why it's a "wide" transformation:


Narrow transformation (map, filter, withColumn) — each output partition depends on exactly ONE input partition. No network transfer needed.
Wide transformation (groupBy, join, distinct, repartition) — each output partition depends on MULTIPLE input partitions. Data must cross executor boundaries.

## Question 12

In [ ]:
from pyspark.sql import functions as F

# Method 1 — explicit null and empty string check
df_clean = df.filter(
    F.col("email").isNotNull() &
    (F.col("username") != "")
)

# Method 2 — handle both null AND empty string on username using trim
df_clean = df.filter(
    F.col("email").isNotNull() &
    (F.trim(F.col("username")) != "")
)

# Method 3 — replace empty strings with null first, then drop
df_clean = (
    df
    .replace("", None, subset=["username"])
    .na.drop(subset=["email", "username"])
)

df_clean.show()

## Question 13

In [ ]:
from pyspark.sql import functions as F

# Calculate min, max, and mean of price in one .agg() call
result = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("mean_price"),
    F.stddev("price").alias("stddev_price"),
    F.count("price").alias("count_non_null")
)

result.show()
# +---------+---------+----------+------------+--------------+
# |min_price|max_price|mean_price|stddev_price|count_non_null|
# +---------+---------+----------+------------+--------------+
# |     10.0|   9999.0|     450.2|      230.15|          5000|
# +---------+---------+----------+------------+--------------+

# Combined with groupBy — stats per category
result_grouped = (
    df
    .groupBy("product_category")
    .agg(
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.avg("price").alias("mean_price")
    )
)

result_grouped.show()


## Question 14

In [ ]:
# Source CSV has inconsistent date formats:
# Row 1:  "2024-01-15"       → Spark infers StringType or DateType
# Row 100: "15/01/2024"      → does not match inferred type
# Row 500: "Jan 15 2024"     → does not match
# Row 800: "1705276800"      → Unix timestamp as integer

df = spark.read.option("inferSchema", "true").csv("messy_data.csv")
# Spark samples rows 1–100, sees "2024-01-15", infers DateType
# Row 100 with "15/01/2024" → parsing FAILS silently → becomes null

## Question 15

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("FinalPipeline").getOrCreate()

# Sample raw data
raw_data = [
    (1, "store_A", 250.0),
    (1, "store_A", 250.0),   # duplicate
    (2, "store_B", None),    # null price
    (3, "store_A", 400.0),
    (4, "store_B", 150.0),
    (5, "store_C", None),    # null price
    (6, "store_C", 600.0),
]

df = spark.createDataFrame(raw_data, ["transaction_id", "store_id", "price"])

# Final processing pipeline
df_result = (
    df
    # Step 1: Remove duplicate rows (all columns)
    .dropDuplicates()

    # Step 2: Fill null prices with 0
    .na.fill(0, subset=["price"])

    # Step 3: Group by store_id and calculate total revenue
    .groupBy("store_id")
    .agg(F.sum("price").alias("total_revenue"))
    .orderBy("total_revenue", ascending=False)
)

df_result.show()
# +--------+-------------+
# |store_id|total_revenue|
# +--------+-------------+
# | store_C|        600.0|
# | store_A|        650.0|
# | store_B|        150.0|
# +--------+-------------+

# Optional — write result to Parquet
df_result.write.mode("overwrite").parquet("output/store_revenue/")